In [20]:
from pathlib import Path
import json
import re

EXCERPT_ID = "excerpt_000"

# quantification.ipynb:
# storyfx/src/playback_scheduling/quantification.ipynb

PROJECT_ROOT = Path.cwd().parents[1]

EXCERPT_DIR = PROJECT_ROOT / "data" / EXCERPT_ID

LABELSTUDIO_DIR = EXCERPT_DIR / "label_studio"
ALIGNMENT_DIR = EXCERPT_DIR / "alignment"
ANNOTATIONS_DIR = EXCERPT_DIR / "annotations"

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# INPUTS
# ============================================================

FA_FILE = (
    ALIGNMENT_DIR /
    f"{EXCERPT_ID}_fa_parsed.txt"
)

SEGMENTS_LS_FILE = (
    LABELSTUDIO_DIR /
    f"{EXCERPT_ID}_segments_ls.json"
)

DISCRETE_SFX_LS_FILE = (
    LABELSTUDIO_DIR /
    f"{EXCERPT_ID}_discfx_ls.json"
)

AMBIENT_SFX_LS_FILE = (
    LABELSTUDIO_DIR /
    f"{EXCERPT_ID}_ambfx_ls.json"
)


# ============================================================
# OUTPUTS
# ============================================================

SEGMENTS_FINAL_FILE = (
    ANNOTATIONS_DIR /
    f"{EXCERPT_ID}_segments_final.json"
)

DISCRETE_SFX_FINAL_FILE = (
    ANNOTATIONS_DIR /
    f"{EXCERPT_ID}_discrete_sfx_final.json"
)

AMBIENT_SFX_FINAL_FILE = (
    ANNOTATIONS_DIR /
    f"{EXCERPT_ID}_ambient_sfx_final.json"
)


print("Project root:", PROJECT_ROOT)
print("Excerpt:", EXCERPT_ID)

print("\nFA:")
print(FA_FILE)

print("\nLabel Studio:")
print(SEGMENTS_LS_FILE)
print(DISCRETE_SFX_LS_FILE)
print(AMBIENT_SFX_LS_FILE)

print("\nFinal annotations:")
print(SEGMENTS_FINAL_FILE)
print(DISCRETE_SFX_FINAL_FILE)
print(AMBIENT_SFX_FINAL_FILE)

Project root: C:\Users\gorda\Desktop\StoryFX\storyfx
Excerpt: excerpt_000

FA:
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt

Label Studio:
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\label_studio\excerpt_000_segments_ls.json
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\label_studio\excerpt_000_discfx_ls.json
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\label_studio\excerpt_000_ambfx_ls.json

Final annotations:
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\annotations\excerpt_000_segments_final.json
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\annotations\excerpt_000_discrete_sfx_final.json
C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\annotations\excerpt_000_ambient_sfx_final.json


In [21]:
def clean_text(text):
    """
    Normalize text for comparison with MFA tokens.
    """

    if not text:
        return ""

    if isinstance(text, list):
        text = " ".join(text)

    text = text.replace("\\n", " ").replace("\n", " ")

    # MFA may separate compounds
    text = text.replace("-", " ")

    # Lowercase + remove punctuation
    text = re.sub(
        r"[^\w\s]",
        "",
        text.lower()
    )

    return " ".join(text.split())


def load_fa_words(fa_path):
    """
    Load parsed MFA representation:

        word|0.42 another|0.81 ...

    Returns ordered words with their MFA end timestamps.
    """

    with open(fa_path, "r", encoding="utf-8") as f:
        content = f.read().strip()

    fa_words = []

    for token in content.split():

        word, timestamp = token.rsplit("|", 1)

        cleaned_word = clean_text(word)

        if cleaned_word:
            fa_words.append({
                "word": cleaned_word,
                "end": float(timestamp)
            })

    return fa_words


fa_words = load_fa_words(FA_FILE)

print(f"Loaded {len(fa_words)} MFA words.")
print(fa_words[:10])

Loaded 545 MFA words.
[{'word': 'why', 'end': 0.93}, {'word': 'they', 'end': 1.07}, {'word': 'came', 'end': 1.38}, {'word': 'east', 'end': 1.66}, {'word': 'i', 'end': 1.77}, {'word': 'dont', 'end': 1.94}, {'word': 'know', 'end': 2.36}, {'word': 'they', 'end': 3.33}, {'word': 'had', 'end': 3.46}, {'word': 'spent', 'end': 3.68}]


In [22]:
from difflib import SequenceMatcher


def find_text_span(
    region_text,
    fa_words,
    search_start_idx=0,
    anchor_size=5,
    fuzzy_threshold=0.80
):
    """
    Resolve a Label Studio text span against MFA words.

    Strategy:
    1. Exact head/tail anchor matching.
    2. If exact matching fails, use fuzzy token matching.
    3. Preserve chronological search order where possible.

    Returns:
        {
            "start_anchor_time": float,
            "end_anchor_time": float,
            "start_word_index": int,
            "end_word_index": int,
            "next_search_index": int
        }
    """

    target_words = clean_text(region_text).split()

    if not target_words:
        return None

    # --------------------------------------------------------
    # Phrase matcher
    # --------------------------------------------------------

    def find_phrase(
        phrase,
        start_idx,
        allow_fuzzy=True
    ):
        n = len(phrase)

        # ================================================
        # 1. EXACT MATCH
        # ================================================

        for i in range(
            start_idx,
            len(fa_words) - n + 1
        ):

            candidate = [
                fa_words[i + j]["word"]
                for j in range(n)
            ]

            if candidate == phrase:
                return i, i + n - 1


        # ================================================
        # 2. FUZZY MATCH
        # ================================================

        if not allow_fuzzy:
            return None

        best_match = None
        best_score = 0.0

        # Allow MFA sequence to be slightly shorter/longer
        # than the source phrase.
        min_window = max(1, n - 2)
        max_window = n + 2

        target_string = " ".join(phrase)

        for window_size in range(
            min_window,
            max_window + 1
        ):

            for i in range(
                start_idx,
                len(fa_words) - window_size + 1
            ):

                candidate_words = [
                    fa_words[i + j]["word"]
                    for j in range(window_size)
                ]

                candidate_string = " ".join(
                    candidate_words
                )

                score = SequenceMatcher(
                    None,
                    target_string,
                    candidate_string
                ).ratio()

                if score > best_score:

                    best_score = score

                    best_match = (
                        i,
                        i + window_size - 1
                    )


        if (
            best_match is not None
            and best_score >= fuzzy_threshold
        ):

            print(
                f"⚠️ Fuzzy FA match "
                f"({best_score:.2f}): "
                f"'{' '.join(phrase)}'"
            )

            return best_match


        return None


    # ========================================================
    # SHORT SEGMENTS
    # ========================================================

    # For small segments it is safer to match the whole thing.
    if len(target_words) <= 10:

        match = find_phrase(
            target_words,
            search_start_idx
        )

        # Global fallback
        if match is None and search_start_idx > 0:

            match = find_phrase(
                target_words,
                0
            )

        if match is None:
            return None

        start_idx, end_idx = match


    # ========================================================
    # LONG SEGMENTS
    # ========================================================

    else:

        k = min(
            anchor_size,
            len(target_words)
        )

        head = target_words[:k]
        tail = target_words[-k:]


        # ----------------------------
        # Find head
        # ----------------------------

        head_match = find_phrase(
            head,
            search_start_idx
        )

        if (
            head_match is None
            and search_start_idx > 0
        ):

            head_match = find_phrase(
                head,
                0
            )


        if head_match is None:
            return None


        start_idx = head_match[0]


        # ----------------------------
        # Find tail
        # ----------------------------

        tail_search_start = (
            head_match[1] + 1
        )

        tail_match = find_phrase(
            tail,
            tail_search_start
        )


        if tail_match is None:
            return None


        end_idx = tail_match[1]


    # ========================================================
    # RESULT
    # ========================================================

    return {
        "start_anchor_time":
            fa_words[start_idx]["end"],

        "end_anchor_time":
            fa_words[end_idx]["end"],

        "start_word_index":
            start_idx,

        "end_word_index":
            end_idx,

        "next_search_index":
            end_idx + 1
    }

In [23]:
def quantify_segments(
    labelstudio_path,
    fa_words,
    excerpt_id
):
    """
    Convert Label Studio segment annotations into canonical
    StoryFX segment annotations with MFA time anchors.
    """

    with open(
        labelstudio_path,
        "r",
        encoding="utf-8"
    ) as f:
        ls_data = json.load(f)

    raw_segments = []


    # ========================================================
    # EXTRACT LABEL STUDIO REGIONS
    # ========================================================

    for item in ls_data:

        for annotation in item.get("annotations", []):

            regions = {}

            for result in annotation.get("result", []):

                region_id = result.get("id")

                if not region_id:
                    continue

                if region_id not in regions:
                    regions[region_id] = {
                        "labelstudio_id": region_id,
                        "text": "",
                        "labels": [],
                        "choices": []
                    }

                value = result.get("value", {})
                from_name = result.get("from_name")


                # Text span
                if (
                    "text" in value
                    and not isinstance(value["text"], list)
                ):
                    regions[region_id]["text"] = (
                        value["text"]
                    )


                # Segment type
                #
                # Supports both your newer terminology and
                # legacy Label Studio configurations.
                if from_name in {"segment", "scene"}:
                    regions[region_id]["labels"].extend(
                        value.get("labels", [])
                    )


                # Reverb profile
                elif from_name in {
                    "reverb_profile",
                    "spatial_profile"
                }:
                    choices = value.get("choices", [])

                    if choices:
                        regions[region_id]["choices"] = (
                            choices
                        )


            raw_segments.extend(
                region
                for region in regions.values()
                if region["text"]
            )


    # ========================================================
    # QUANTIFY IN CHRONOLOGICAL ORDER
    # ========================================================

    quantified = []

    current_search_index = 0

    for region in raw_segments:

        match = find_text_span(
            region["text"],
            fa_words,
            current_search_index
        )

        if match is None:
            raise ValueError(
                "Could not align segment:\n"
                f"{region['text'][:100]}"
            )

        current_search_index = (
            match["next_search_index"]
        )

        quantified.append({
            "labelstudio_id":
                region["labelstudio_id"],

            "segment_type":
                region["labels"][0]
                if region["labels"]
                else None,

            "start_anchor_time":
                match["start_anchor_time"],

            "end_anchor_time":
                match["end_anchor_time"],

            "reverb_profile":
                region["choices"][0]
                if region["choices"]
                else None
        })


    # ========================================================
    # CREATE CANONICAL STORYFX IDS
    # ========================================================

    for i, segment in enumerate(quantified):

        segment["segment_id"] = (
            f"{excerpt_id}_segment_{i:03d}"
        )


    # Put primary ID first just for readability
    quantified = [
        {
            "segment_id":
                segment["segment_id"],

            "labelstudio_id":
                segment["labelstudio_id"],

            "segment_type":
                segment["segment_type"],

            "start_anchor_time":
                segment["start_anchor_time"],

            "end_anchor_time":
                segment["end_anchor_time"],

            "reverb_profile":
                segment["reverb_profile"]
        }

        for segment in quantified
    ]


    return quantified

In [25]:
segments_final = quantify_segments(
    SEGMENTS_LS_FILE,
    fa_words,
    EXCERPT_ID
)

with open(
    SEGMENTS_FINAL_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        segments_final,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    f"Saved {len(segments_final)} segments."
)

print(
    json.dumps(
        segments_final,
        indent=2
    )
)

Saved 4 segments.
[
  {
    "segment_id": "excerpt_000_segment_000",
    "labelstudio_id": "E46tUcDStm",
    "segment_type": "expository",
    "start_anchor_time": 0.93,
    "end_anchor_time": 29.24,
    "reverb_profile": null
  },
  {
    "segment_id": "excerpt_000_segment_001",
    "labelstudio_id": "tr5rdLxbcu",
    "segment_type": "dramatized",
    "start_anchor_time": 30.65,
    "end_anchor_time": 132.19,
    "reverb_profile": "outdoor_open"
  },
  {
    "segment_id": "excerpt_000_segment_002",
    "labelstudio_id": "CARw-j7Pd-",
    "segment_type": "dramatized",
    "start_anchor_time": 133.34,
    "end_anchor_time": 134.96,
    "reverb_profile": "indoor_medium_reflective_elongated"
  },
  {
    "segment_id": "excerpt_000_segment_003",
    "labelstudio_id": "3hFCNy4Hk6",
    "segment_type": "spatial_extended",
    "start_anchor_time": 135.64,
    "end_anchor_time": 204.55,
    "reverb_profile": "indoor_medium_absorbed_regular"
  }
]
